# Track A 与 Track B 效果对比 / Track A vs Track B Comparison

这个 notebook 把 `Track A` 的 Gaussian HMM 和 `Track B` 的 Transformer + clustering 放到同一份数据、同一段样本里比较。  
This notebook compares the `Track A` Gaussian HMM with the `Track B` Transformer + clustering on the same data and sample window.

我们重点看三件事：  
We focus on three things:

1. 时间轴上，两条方法对市场状态的分段是否相似。  
   Whether the two tracks segment market regimes similarly over time.
2. 聚类结果和 HMM 状态在标签层面有多接近。  
   How closely the cluster assignments line up with the HMM states.
3. Track B 的训练曲线，以及它和 Track A 的输出在可解释性上是否一致。  
   The Track B training curve, and whether its outputs remain interpretable relative to Track A.


In [ ]:
import itertools
from dataclasses import asdict

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from track_b_pipeline import TrackBConfig, run_track_b, shade_segments

plt.style.use("ggplot")

# 为图表设置中文字体回退，避免标题、图例和坐标轴里的中文显示成方框或空白。
# Set up Chinese font fallbacks so titles, legends, and axes labels render correctly.
available_fonts = {font.name for font in fm.fontManager.ttflist}
preferred_fonts = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = [font_name for font_name in preferred_fonts if font_name in available_fonts] or ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


## 参数设置 / Configuration

这里统一设置 Track B 的主参数。Track A 的 HMM 参考标签会在 `run_track_b(...)` 里一起生成，所以两边天然是同口径的 apples-to-apples 比较。  
Set the main Track B hyperparameters here. The Track A HMM reference labels are generated inside `run_track_b(...)`, which makes the comparison apples-to-apples by construction.


In [ ]:
# 在这里集中调整 Track B 的主要参数；Track A 会自动复用同一份样本区间和参考 HMM。
# Adjust the main Track B hyperparameters here; Track A will automatically reuse the same sample window and reference HMM.
track_b_config = TrackBConfig(
    start_date="2015-01-01",
    window_size=60,
    mask_ratio=0.20,
    batch_size=64,
    num_epochs=200,
    learning_rate=1e-4,
    d_model=128,
    embedding_dim=64,
    n_heads=4,
    num_layers=2,
    dropout=0.10,
    early_stopping_patience=20,
    min_epochs=5,
    log_every=1,
)

display(pd.Series(asdict(track_b_config), name="value / 数值").to_frame())


## 辅助函数 / Helper Functions

下面这些函数负责把 Track B 的 cluster 对齐到 Track A 的 HMM state，并生成对比图和汇总表。  
The helper functions below align Track B clusters to Track A HMM states, then build the comparison charts and summary tables.


In [ ]:
PALETTE = ["#b2182b", "#fdae61", "#67a9cf", "#1a9850", "#762a83", "#5e3c99"]


def build_state_colors(ordered_states):
    # 用固定调色盘给 HMM 状态上色，方便和 Track B 的映射结果共用颜色。
    # Use a fixed palette for HMM states so Track B can reuse the same semantic colors after mapping.
    return {state: PALETTE[i % len(PALETTE)] for i, state in enumerate(ordered_states)}



def best_cluster_mapping(window_reference: pd.DataFrame):
    # 穷举所有 cluster -> state 的一一映射，找到 matched accuracy 最高的那种对齐方式。
    # Enumerate all one-to-one cluster -> state mappings and keep the alignment with the best matched accuracy.
    hmm_states = sorted(window_reference["hmm_state"].unique())
    clusters = sorted(window_reference["cluster"].unique())

    best_mapping = None
    best_accuracy = -1.0

    for perm in itertools.permutations(hmm_states, len(clusters)):
        mapping = dict(zip(clusters, perm))
        mapped_states = window_reference["cluster"].map(mapping)
        accuracy = float((mapped_states.to_numpy() == window_reference["hmm_state"].to_numpy()).mean())
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_mapping = mapping

    return best_mapping, best_accuracy



def safe_silhouette(matrix: np.ndarray, labels: np.ndarray):
    # silhouette 至少需要两个簇，而且每个簇里不能只剩一个样本；不满足时就返回 NaN。
    # Silhouette needs at least two groups and more than one sample per group; otherwise return NaN.
    unique_labels, counts = np.unique(labels, return_counts=True)
    if len(unique_labels) < 2 or np.any(counts < 2):
        return np.nan
    return float(silhouette_score(matrix, labels))



def build_pca_view(results: dict, comparison_df: pd.DataFrame):
    # 把 Track B 的高维 embedding 压到 2 个主成分里，便于做更轻量的直观可视化比较。
    # Compress the high-dimensional Track B embeddings into 2 principal components for a lighter-weight visual comparison.
    pca = PCA(n_components=2, random_state=results["config"].random_state)
    coords = pca.fit_transform(results["embeddings"])

    pca_view_df = pd.DataFrame(
        {
            "pc1": coords[:, 0],
            "pc2": coords[:, 1],
            "cluster": comparison_df["cluster"].to_numpy(),
            "hmm_state": comparison_df["hmm_state"].to_numpy(),
            "mapped_state": comparison_df["mapped_state"].to_numpy(),
            "match_flag": comparison_df["match_flag"].to_numpy(),
            "hmm_regime_cn": comparison_df["hmm_regime_cn"].to_numpy(),
            "hmm_regime_en": comparison_df["hmm_regime_en"].to_numpy(),
            "mapped_regime_cn": comparison_df["mapped_regime_cn"].to_numpy(),
            "mapped_regime_en": comparison_df["mapped_regime_en"].to_numpy(),
        }
    )

    return pca_view_df, pca.explained_variance_ratio_



def prepare_comparison_outputs(results: dict):
    # 把 Track B 的 cluster 映射到 Track A 的 state，再整理出后面展示要用的表格。
    # Map Track B clusters onto Track A states, then prepare the tables used in the downstream displays.
    comparison_df = results["window_reference"].copy()
    best_mapping, matched_accuracy = best_cluster_mapping(comparison_df)

    comparison_df["mapped_state"] = comparison_df["cluster"].map(best_mapping)
    comparison_df["mapped_regime_cn"] = comparison_df["mapped_state"].map(results["label_map_cn"])
    comparison_df["mapped_regime_en"] = comparison_df["mapped_state"].map(results["label_map_en"])
    comparison_df["match_flag"] = comparison_df["mapped_state"] == comparison_df["hmm_state"]
    comparison_df["rolling_agreement_63d"] = comparison_df["match_flag"].rolling(63, min_periods=10).mean()

    pca_view_df, pca_explained_ratio = build_pca_view(results, comparison_df)
    embedding_matrix = results["embeddings"]
    pca_matrix = pca_view_df[["pc1", "pc2"]].to_numpy()

    silhouette_hmm_on_embedding = safe_silhouette(embedding_matrix, comparison_df["hmm_state"].to_numpy())
    silhouette_track_b_on_embedding = safe_silhouette(embedding_matrix, comparison_df["cluster"].to_numpy())
    silhouette_hmm_on_pca2d = safe_silhouette(pca_matrix, comparison_df["hmm_state"].to_numpy())
    silhouette_track_b_on_pca2d = safe_silhouette(pca_matrix, comparison_df["cluster"].to_numpy())

    best_epoch = int(results["history_df"].loc[results["history_df"]["val_loss"].idxmin(), "epoch"])
    metric_summary = pd.DataFrame(
        [
            {
                "sample_start / 样本起点": comparison_df["date"].min().date(),
                "sample_end / 样本终点": comparison_df["date"].max().date(),
                "window_count / 窗口数量": len(comparison_df),
                "track_a_states / Track A 状态数": results["best_n_states"],
                "track_b_clusters / Track B 聚类数": results["target_cluster_count"],
                "best_val_loss / 最佳验证损失": results["history_df"]["val_loss"].min(),
                "best_epoch / 最佳轮数": best_epoch,
                "ARI / 调整兰德指数": results["ari_score"],
                "NMI / 归一化互信息": results["nmi_score"],
                "matched_accuracy / 匹配准确率": matched_accuracy,
                "silhouette_hmm_on_embedding / embedding空间的HMM轮廓系数": silhouette_hmm_on_embedding,
                "silhouette_track_b_on_embedding / embedding空间的Track B轮廓系数": silhouette_track_b_on_embedding,
                "silhouette_hmm_on_pca2d / PCA2D空间的HMM轮廓系数": silhouette_hmm_on_pca2d,
                "silhouette_track_b_on_pca2d / PCA2D空间的Track B轮廓系数": silhouette_track_b_on_pca2d,
                "pca2d_explained_var / PCA二维解释率": float(pca_explained_ratio.sum()),
            }
        ]
    )

    mapping_table = pd.DataFrame(
        {
            "cluster / 聚类": list(best_mapping.keys()),
            "mapped_state / 对应状态": list(best_mapping.values()),
        }
    ).sort_values("cluster / 聚类")
    mapping_table["mapped_regime_cn / 中文标签"] = mapping_table["mapped_state / 对应状态"].map(results["label_map_cn"])
    mapping_table["mapped_regime_en / English label"] = mapping_table["mapped_state / 对应状态"].map(results["label_map_en"])

    raw_crosstab = pd.crosstab(comparison_df["cluster"], comparison_df["hmm_regime_en"])
    mapped_crosstab = pd.crosstab(comparison_df["mapped_regime_en"], comparison_df["hmm_regime_en"])

    cluster_summary = results["cluster_summary"].copy()
    cluster_summary["mapped_state"] = cluster_summary["cluster"].map(best_mapping)
    cluster_summary["mapped_regime_cn"] = cluster_summary["mapped_state"].map(results["label_map_cn"])
    cluster_summary["mapped_regime_en"] = cluster_summary["mapped_state"].map(results["label_map_en"])
    cluster_summary = cluster_summary.sort_values(["mapped_state", "cluster"]).reset_index(drop=True)

    return comparison_df, metric_summary, mapping_table, raw_crosstab, mapped_crosstab, cluster_summary, pca_view_df, pca_explained_ratio



def draw_annotated_heatmap(ax, frame: pd.DataFrame, title: str, cmap: str = "Blues"):
    # 用简单 heatmap 展示 cluster / regime 对齐关系，并在格子里直接写上计数。
    # Show cluster / regime alignment with a simple heatmap and annotate each cell with the raw count.
    data = frame.to_numpy()
    im = ax.imshow(data, cmap=cmap, aspect="auto")
    ax.set_xticks(range(frame.shape[1]), frame.columns, rotation=25, ha="right")
    ax.set_yticks(range(frame.shape[0]), frame.index)
    ax.set_title(title)

    for i in range(frame.shape[0]):
        for j in range(frame.shape[1]):
            ax.text(j, i, f"{int(data[i, j])}", ha="center", va="center", color="black", fontsize=10)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)



def plot_pca_comparison_2d(results: dict, pca_view_df: pd.DataFrame, pca_explained_ratio: np.ndarray):
    # 在同一套 PCA 二维坐标里，把 Track A 标签和 Track B 标签并排画出来，方便做直观比较。
    # Plot Track A labels and Track B labels side by side in the same PCA-2D coordinates for an intuitive comparison.
    ordered_states = results["ordered_states"]
    state_colors = build_state_colors(ordered_states)
    axis_labels = [
        f"PC1 ({pca_explained_ratio[0]:.1%})",
        f"PC2 ({pca_explained_ratio[1]:.1%})",
    ]

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

    for state in ordered_states:
        mask = pca_view_df["hmm_state"] == state
        ax_left.scatter(
            pca_view_df.loc[mask, "pc1"],
            pca_view_df.loc[mask, "pc2"],
            s=18,
            alpha=0.72,
            color=state_colors[state],
            label=f"state {state}: {results['label_map_cn'][state]} / {results['label_map_en'][state]}",
        )

    for state in ordered_states:
        mask = pca_view_df["mapped_state"] == state
        ax_right.scatter(
            pca_view_df.loc[mask, "pc1"],
            pca_view_df.loc[mask, "pc2"],
            s=18,
            alpha=0.72,
            color=state_colors[state],
            label=f"mapped state {state}: {results['label_map_cn'][state]} / {results['label_map_en'][state]}",
        )

    disagreement_mask = ~pca_view_df["match_flag"]
    if disagreement_mask.any():
        ax_right.scatter(
            pca_view_df.loc[disagreement_mask, "pc1"],
            pca_view_df.loc[disagreement_mask, "pc2"],
            s=28,
            facecolors="none",
            edgecolors="black",
            linewidths=0.7,
            label="disagreement / 不一致",
        )

    ax_left.set_title("PCA 2D with Track A HMM labels / PCA二维下的 Track A 标签")
    ax_right.set_title("PCA 2D with Track B mapped labels / PCA二维下的 Track B 映射标签")

    for ax in [ax_left, ax_right]:
        ax.set_xlabel(axis_labels[0])
        ax.set_ylabel(axis_labels[1])
        ax.grid(True, alpha=0.25)

    ax_right.legend(loc="upper left", bbox_to_anchor=(1.02, 0.98), fontsize=8)
    plt.tight_layout()
    plt.show()



def plot_disagreement_focus_2d(results: dict, pca_view_df: pd.DataFrame, pca_explained_ratio: np.ndarray):
    # 单独把 disagreement 窗口拎出来看，并用浅灰背景保留整体分布参照。
    # Isolate disagreement windows in their own figure while keeping the full distribution as a light-gray reference.
    ordered_states = results["ordered_states"]
    state_colors = build_state_colors(ordered_states)
    disagreement_df = pca_view_df.loc[~pca_view_df["match_flag"]].copy()
    axis_labels = [
        f"PC1 ({pca_explained_ratio[0]:.1%})",
        f"PC2 ({pca_explained_ratio[1]:.1%})",
    ]

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

    for ax in [ax_left, ax_right]:
        ax.scatter(
            pca_view_df["pc1"],
            pca_view_df["pc2"],
            s=10,
            alpha=0.18,
            color="#b0b0b0",
            label="all windows / 全部窗口",
        )

    if disagreement_df.empty:
        for ax in [ax_left, ax_right]:
            ax.text(0.5, 0.5, "No disagreement / 没有不一致窗口", transform=ax.transAxes, ha="center", va="center", fontsize=12)
    else:
        for state in ordered_states:
            hmm_mask = disagreement_df["hmm_state"] == state
            if hmm_mask.any():
                ax_left.scatter(
                    disagreement_df.loc[hmm_mask, "pc1"],
                    disagreement_df.loc[hmm_mask, "pc2"],
                    s=34,
                    alpha=0.88,
                    color=state_colors[state],
                    label=f"Track A state {state}: {results['label_map_cn'][state]} / {results['label_map_en'][state]}",
                )

        for state in ordered_states:
            mapped_mask = disagreement_df["mapped_state"] == state
            if mapped_mask.any():
                ax_right.scatter(
                    disagreement_df.loc[mapped_mask, "pc1"],
                    disagreement_df.loc[mapped_mask, "pc2"],
                    s=34,
                    alpha=0.88,
                    color=state_colors[state],
                    label=f"Track B mapped state {state}: {results['label_map_cn'][state]} / {results['label_map_en'][state]}",
                )

    ax_left.set_title("Disagreement windows by Track A labels / 按 Track A 标签看的不一致窗口")
    ax_right.set_title("Disagreement windows by Track B mapped labels / 按 Track B 映射标签看的不一致窗口")

    for ax in [ax_left, ax_right]:
        ax.set_xlabel(axis_labels[0])
        ax.set_ylabel(axis_labels[1])
        ax.grid(True, alpha=0.25)

    ax_left.legend(loc="upper left", fontsize=8)
    ax_right.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_comparison_dashboard(results: dict, comparison_df: pd.DataFrame, raw_crosstab: pd.DataFrame, mapped_crosstab: pd.DataFrame):
    # 先画时间轴上的直观对比，再画训练曲线、BIC 和标签对齐的统计图。
    # First show the intuitive time-series comparison, then draw training curves, BIC, and label-alignment diagnostics.
    ordered_states = results["ordered_states"]
    state_colors = build_state_colors(ordered_states)

    aligned_close = results["close"].rename(columns={"^VIX": "VIX"}).loc[comparison_df["date"]]
    hmm_state_series = comparison_df.set_index("date")["hmm_state"]
    mapped_state_series = comparison_df.set_index("date")["mapped_state"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

    axes[0, 0].plot(aligned_close.index, aligned_close["SPY"], color="black", linewidth=1.35)
    shade_segments(axes[0, 0], aligned_close.index, hmm_state_series.values, state_colors, alpha=0.18)
    axes[0, 0].set_title("Track A regimes on SPY / Track A 状态与 SPY")
    axes[0, 0].set_ylabel("Adjusted close / 复权收盘价")

    axes[0, 1].plot(aligned_close.index, aligned_close["SPY"], color="black", linewidth=1.35)
    shade_segments(axes[0, 1], aligned_close.index, mapped_state_series.values, state_colors, alpha=0.18)
    axes[0, 1].set_title("Track B mapped clusters on SPY / Track B 映射后聚类与 SPY")

    axes[1, 0].plot(aligned_close.index, aligned_close["VIX"], color="#4c72b0", linewidth=1.20)
    shade_segments(axes[1, 0], aligned_close.index, hmm_state_series.values, state_colors, alpha=0.18)
    axes[1, 0].set_title("Track A regimes on VIX / Track A 状态与 VIX")
    axes[1, 0].set_ylabel("Index level / 指数水平")

    axes[1, 1].plot(aligned_close.index, aligned_close["VIX"], color="#4c72b0", linewidth=1.20)
    shade_segments(axes[1, 1], aligned_close.index, mapped_state_series.values, state_colors, alpha=0.18)
    axes[1, 1].set_title("Track B mapped clusters on VIX / Track B 映射后聚类与 VIX")

    legend_handles = [
        plt.Line2D([0], [0], color=state_colors[state], linewidth=6, alpha=0.7, label=f"state {state}: {results['label_map_cn'][state]} / {results['label_map_en'][state]}")
        for state in ordered_states
    ]
    axes[0, 1].legend(handles=legend_handles, loc="upper left", fontsize=8)

    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    axes[0, 0].plot(results["bic_summary"]["n_states"], results["bic_summary"]["bic"], marker="o", color="#c44e52")
    axes[0, 0].axvline(results["best_n_states"], color="black", linestyle="--", alpha=0.7)
    axes[0, 0].set_title("Track A BIC scan / Track A 的 BIC 扫描")
    axes[0, 0].set_xlabel("State count / 状态数量")
    axes[0, 0].set_ylabel("BIC")

    axes[0, 1].plot(results["history_df"]["epoch"], results["history_df"]["train_loss"], marker="o", label="train")
    axes[0, 1].plot(results["history_df"]["epoch"], results["history_df"]["val_loss"], marker="o", label="val")
    axes[0, 1].set_title("Track B training curve / Track B 训练曲线")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Masked MSE / 遮罩 MSE")
    axes[0, 1].legend()

    draw_annotated_heatmap(axes[1, 0], raw_crosstab, "Raw cluster vs HMM regime / 原始 cluster 与 HMM 标签")
    draw_annotated_heatmap(axes[1, 1], mapped_crosstab, "Mapped cluster vs HMM regime / 映射后 cluster 与 HMM 标签", cmap="Greens")

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(comparison_df["date"], comparison_df["rolling_agreement_63d"], color="#2a9d8f", linewidth=1.5)
    ax.axhline(comparison_df["match_flag"].mean(), color="black", linestyle="--", alpha=0.7, label="overall agreement / 总体一致率")
    ax.set_title("63-day rolling agreement between Track A and Track B / Track A 与 Track B 的 63 日滚动一致率")
    ax.set_ylabel("Agreement / 一致率")
    ax.set_ylim(0.0, 1.02)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 运行比较 / Run The Comparison

这一格会真正运行 Track B，同时顺带得到 Track A 的 HMM 参考状态。由于会下载 Yahoo 和 FRED 数据、再训练 Transformer，所以运行时间会比前面的 demo notebook 长一些。  
This cell runs Track B end-to-end and also produces the Track A HMM reference states. It takes longer than the earlier demo notebooks because it downloads Yahoo/FRED data and trains the Transformer.


In [ ]:
# 运行 Track B，并把 Track A / Track B 的对比表、PCA 视图和可视化一并准备好。
# Run Track B, then prepare the Track A / Track B comparison tables, PCA view, and visual outputs.
results = run_track_b(track_b_config)
comparison_df, metric_summary, mapping_table, raw_crosstab, mapped_crosstab, cluster_summary, pca_view_df, pca_explained_ratio = prepare_comparison_outputs(results)

print("Common sample starts / 共同样本起点:", results["hmm_features"].index.min().date())
print("Common sample ends / 共同样本终点:", results["hmm_features"].index.max().date())
print("Track A best state count / Track A 最优状态数:", results["best_n_states"])
print("Track B cluster count / Track B 聚类数量:", results["target_cluster_count"])
print("PCA 2D explained variance / PCA 二维解释率:", round(float(pca_explained_ratio.sum()), 4))


## 关键汇总表 / Key Summary Tables

先看一眼几个最核心的指标，再看 Track A 的状态解释表和 Track B 的聚类解释表。  
Start with the core metrics, then inspect the Track A state interpretation table and the Track B cluster summary table.


In [ ]:
# 先展示总指标，再展示 cluster -> state 的映射，以及两条方法各自的解释表。
# Show the headline metrics first, then the cluster-to-state mapping, followed by the interpretation tables for both tracks.
display(metric_summary.round(4))
display(mapping_table)

display(
    results["hmm_state_summary"][[
        "hmm_state",
        "hmm_regime_cn",
        "hmm_regime_en",
        "n_days",
        "share",
        "avg_SPY_ret",
        "avg_SPY_vol_21d",
        "avg_VIX_level",
        "avg_credit_spread_proxy",
        "avg_curve_slope_10y2y",
    ]].round(4)
)

display(
    cluster_summary[[
        "cluster",
        "mapped_state",
        "mapped_regime_cn",
        "mapped_regime_en",
        "n_windows",
        "window_share",
        "avg_SPY_ret",
        "avg_VIX_level",
        "avg_credit_spread_proxy",
        "avg_curve_slope_10y2y",
    ]].round(4)
)


## PCA 二维直观对比 / PCA 2D Visual Comparison

可以把 Track B 的高维 embedding 先压到 2 个主成分里，再在同一套坐标上并排画两张图：左边按 Track A 的 HMM 标签上色，右边按 Track B 映射后的标签上色。  
We can first compress the high-dimensional Track B embeddings into 2 principal components, then draw two side-by-side plots in the same coordinates: the left plot is colored by Track A HMM labels, and the right plot is colored by the mapped Track B labels.

这样看有两个好处：  
This helps in two ways:

1. 如果两张图里的色块出现在相近区域，说明两种方法对样本结构的理解比较一致。  
   If similarly colored regions appear in similar locations across the two plots, the two methods are interpreting the sample structure in a similar way.
2. 如果右图里黑色描边的不一致点很多，说明 Track B 和 Track A 仍然有一批明显分歧的窗口。  
   If the right plot contains many black-outlined disagreement points, Track B and Track A still disagree on a meaningful set of windows.


In [ ]:
# 先看同一套 PCA 二维空间下，Track A 标签和 Track B 标签是否把样本切在相近区域。
# First inspect whether Track A labels and Track B labels carve the samples into similar regions in the same PCA-2D space.
plot_pca_comparison_2d(results, pca_view_df, pca_explained_ratio)

# 再把 disagreement 窗口单独拿出来看，确认分歧主要集中在哪些区域。
# Then isolate the disagreement windows to see where the main mismatches are concentrated.
plot_disagreement_focus_2d(results, pca_view_df, pca_explained_ratio)

# 最后看时间序列、训练曲线和标签对齐情况。
# Finally inspect the time-series view, the training curve, and the label-alignment diagnostics.
plot_comparison_dashboard(results, comparison_df, raw_crosstab, mapped_crosstab)


## 读图提示 / Reading Guide

- 如果 `mapped cluster vs HMM regime` 的热力图明显更接近对角线，说明 Track B 学到的 cluster 和 Track A 的 regime 在语义上更接近。  
  If the `mapped cluster vs HMM regime` heatmap is closer to diagonal, Track B's clusters are semantically closer to Track A's regimes.
- 如果 `PCA 2D` 左右两张图里，同色点云大体落在相近区域，说明两种方法在结构上更一致。  
  If similarly colored point clouds occupy similar regions in the two `PCA 2D` plots, the two methods are structurally more aligned.
- 如果 `disagreement` 专门图里，高亮点只集中在少数边界区域，说明两种方法主要是在 regime 过渡带上有分歧。  
  If the dedicated `disagreement` figure highlights only a few boundary regions, the two methods are mainly disagreeing around regime transition zones.
- 如果 `rolling agreement` 在市场压力期明显上升，说明两种方法在 stress period 的识别更一致。  
  If the `rolling agreement` line rises during stress periods, the two methods are more aligned when the market is under pressure.
- 如果 `val loss` 在训练曲线里持续下降，但 `ARI / NMI / matched_accuracy` 没同步上升，说明 Track B 更像是把重建任务学好了，但未必把 regime 语义学得更好。  
  If `val loss` keeps falling while `ARI / NMI / matched_accuracy` do not improve in tandem, Track B may be learning the reconstruction task better without necessarily learning regime semantics better.
